## 실습 1. 개선 데이터 확인하기

 

먼저 모델을 바꾸기 전에 데이터를 확인합니다.

In [1]:
import pandas as pd

df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

print(df.shape)

print(df.columns)

display(df.head())

(5672, 12)
Index(['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물',
       '출판사', '발행(출시)일자', '분야'],
      dtype='str')


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설
3,9791175772892,S000220307771,테오,"21,000","18,900",10%,5%,"1,050",앨런 레비,오팬하우스,20260701,소설
4,9791198547514,S000211748790,수족관,"17,700","15,930",10%,5%,880,유래혁,포스터샵,20240111,소설


### 실행 결과 요약

- 데이터 파일: `books_improved.csv`
- 데이터 크기: **5,672행 × 12열**
- 전체 도서 수: **5,672권**
- 컬럼 수: **12개**
- 도서 제목 컬럼: `상품명`
- 카테고리 컬럼: `분야`
- 주요 정보: `상품코드`, `판매상품 ID`, `상품명`, `정가`, `판매가`, `할인율`, `적립율`, `적립예정포인트`, `인물`, `출판사`, `발행(출시)일자`, `분야`
- 결과 확인: 기존 데이터보다 더 많은 도서 데이터를 확보했으며, 이후 도서 카테고리 분류 모델 학습에 활용할 수 있다.

## 실습 2. Baseline을 먼저 측정하기

 

개선 효과를 말하려면 먼저 개선 전 기준점(Baseline) 이 필요합니다.

 

이번 Baseline은 Chapter 04에서 사용한 구조를 유지합니다.

In [2]:
# 실습 2. Baseline 측정
# 도서 제목 → TF-IDF → Multinomial Naive Bayes → 분야 예측

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 1. 입력값과 정답 분리
X = df["상품명"]
y = df["분야"]

# 2. Train / Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 3. TF-IDF 벡터화
# 훈련 데이터에만 fit
baseline_vectorizer = TfidfVectorizer()

X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

# 4. Multinomial Naive Bayes 학습
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)

# 5. 예측
baseline_pred = baseline_model.predict(X_test_vec)

# 6. 평가
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_macro_f1 = f1_score(
    y_test,
    baseline_pred,
    average="macro"
)

correct_count = (y_test == baseline_pred).sum()
total_count = len(y_test)

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Accuracy(%): {baseline_accuracy * 100:.2f}%")
print(f"Macro F1: {baseline_macro_f1:.4f}")
print(f"정답: {correct_count} / {total_count}")

print("\nClassification Report")
print(
    classification_report(
        y_test,
        baseline_pred,
        zero_division=0
    )
)

Baseline Accuracy: 0.5374
Baseline Accuracy(%): 53.74%
Macro F1: 0.3885
정답: 610 / 1135

Classification Report
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       0.80      0.08      0.14        53
       경제/경영       0.58      0.62      0.60       200
          소설       0.40      0.78      0.53       200
       시/에세이       0.56      0.57      0.57       200
          요리       1.00      0.05      0.10        38
          인문       0.52      0.49      0.51       200
   취미/실용/스포츠       1.00      0.15      0.27        72
      컴퓨터/IT       0.80      0.78      0.79       129

    accuracy                           0.54      1135
   macro avg       0.63      0.39      0.39      1135
weighted avg       0.59      0.54      0.50      1135



### 실행 결과 요약

- 데이터 보강 전 정확도: **약 35%**
- 데이터 보강 후 Baseline Accuracy: **53.74%**
- 정확도 변화: **약 18.74%p 상승**
- Macro F1: **0.3885**
- 테스트 데이터 수: **1,135권**
- 정답 수: **610권**

### 결과 해석

- 기존 약 35%였던 정확도가 **53.74%까지 상승**해 데이터 보강 효과를 확인할 수 있었다.
- 특히 `컴퓨터/IT`는 **F1-score 0.79**로 비교적 높은 성능을 보였다.
- `경제/경영`, `소설`, `시/에세이`, `인문`도 어느 정도 분류가 가능했다.
- 반면 `가정/육아`, `건강`, `요리`, `취미/실용/스포츠`는 recall이 낮아 실제 해당 분야 도서를 많이 놓치고 있다.
- Accuracy는 향상되었지만 **Macro F1이 0.3885**이므로 카테고리별 성능 차이는 여전히 크다.
- 따라서 다음 개선 단계에서는 단순히 데이터 수만 늘리는 것뿐 아니라 **카테고리별 불균형과 TF-IDF 설정, 모델 자체를 함께 개선할 필요가 있다.**

### 최종 판단

데이터를 보강한 뒤 전체 정확도는 **약 35% → 53.74%**로 크게 개선되었다.

하지만 일부 카테고리에 예측이 편중되어 있으므로, 현재 Baseline은 개선 효과를 확인하기 위한 기준점으로 사용하고 이후 실험에서 **Macro F1과 카테고리별 recall을 함께 비교**하는 것이 중요하다.

## 실습 3. 한국어 형태소 분석 적용하기

 

기본 TF-IDF는 제목 문자열을 그대로 사용합니다.

 

이번에는 한국어 제목에서 의미 있는 형태소를 추출해 TF-IDF 입력을 바꿔 봅니다.

In [3]:
# 실습 3. 한국어 형태소 분석 적용하기
# 상품명 → Kiwi 형태소 분석 → NNG, NNP, SL만 추출 → 새 컬럼 생성

from kiwipiepy import Kiwi

# 1. Kiwi 형태소 분석기 생성
kiwi = Kiwi()

# 2. 사용할 품사
# NNG: 일반 명사
# NNP: 고유 명사
# SL : 영문
target_tags = {"NNG", "NNP", "SL"}

# 3. 상품명 전처리 함수
def tokenize_title(text):
    tokens = kiwi.tokenize(str(text))

    selected_tokens = [
        token.form
        for token in tokens
        if token.tag in target_tags
    ]

    return " ".join(selected_tokens)

# 4. 전체 상품명에 적용
df["상품명_토큰"] = df["상품명"].apply(tokenize_title)

# 5. 전처리 전후 확인
display(
    df[["상품명", "상품명_토큰"]].head(10)
)

,상품명,상품명_토큰
0,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항
3,테오,테오
4,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자
6,모순,모순
7,서리꽃 세트,서리 꽃 세트
8,데미안,데미안
9,브람스를 좋아하세요,브람스


### 실행 결과 요약

- 형태소 분석기: `Kiwi`
- 사용 품사: `NNG`, `NNP`, `SL`
- 원본 컬럼: `상품명`
- 생성 컬럼: `상품명_토큰`
- 예시 1: `한국사 이상현상 연구원(일반판)` → `한국사 이상 현상 연구원 일반판`
- 예시 2: `녹색 절벽의 신자들` → `녹색 절벽 신자`
- 예시 3: `브람스를 좋아하세요` → `브람스`
- 결과: 조사나 불필요한 형태소가 줄어들고 핵심 명사 중심의 텍스트로 변환됨
- 다음 단계: `상품명_토큰`을 TF-IDF 입력으로 사용해 Baseline Accuracy **53.74%**와 비교

## 실습 4. 불필요한 단어 필터링하기

In [4]:
# 실습 4. 불필요한 단어 필터링하기

# 최소한의 불용어만 사용
stopwords = {
    "에디션",
}

# 토큰 정제 함수
def clean_tokens(text):
    tokens = str(text).split()

    cleaned_tokens = [
        token
        for token in tokens
        if len(token) >= 2              # 너무 짧은 토큰 제거
        and not token.isdigit()         # 숫자로만 된 토큰 제거
        and token not in stopwords      # 불용어 제거
    ]

    return " ".join(cleaned_tokens)

# 상품명_토큰 컬럼에 정제 적용
df["상품명_정제"] = df["상품명_토큰"].apply(clean_tokens)

# 전처리 전후 확인
display(
    df[["상품명", "상품명_토큰", "상품명_정제"]].head(10)
)

,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스


### 실행 결과 요약

- 정제 기준: 2글자 미만 토큰 제거, 숫자로만 구성된 토큰 제거, 불용어 `에디션` 제거
- 원본 컬럼: `상품명`
- 형태소 분석 컬럼: `상품명_토큰`
- 최종 정제 컬럼: `상품명_정제`

- 전처리 예시 1: `빵 충 사육 준수 사항` → `사육 준수 사항`
- 전처리 예시 2: `서리 꽃 세트` → `서리 세트`
- 유지된 예시: `녹색 절벽 신자`, `브람스`처럼 이미 의미 있는 토큰만 남은 경우에는 그대로 유지되었다.

- 주요 개선 결과: 너무 짧은 토큰과 불필요한 표현을 제거해 제목 표현을 더 간결하게 만들었고, TF-IDF에는 상대적으로 의미 있는 단어가 중심이 되도록 정제했다.
- 주의점: 짧은 단어라고 해서 항상 불필요한 것은 아니므로 과도한 필터링은 피해야 한다.
- 현재 단계의 의미: 전처리 결과는 더 깔끔해졌지만 실제 분류 성능이 향상되었는지는 아직 확인되지 않았다.
- 비교 기준: 기존 Baseline Accuracy **53.74%**, Macro F1 **0.3885**
- 다음 단계: `상품명_정제`를 TF-IDF 입력으로 사용해 Accuracy, Macro F1, 분야별 성능 변화를 비교한다.

## 실습 5. Improved 분류 모델 평가하기

이제 형태소 분석과 단어 필터링을 적용한 텍스트로 다시 TF-IDF와 Naive Bayes를 학습합니다.

 

알고리즘 자체는 바꾸지 않습니다.

In [5]:
# 실습 5. Improved 분류 모델 평가하기
# 형태소 분석 + 단어 필터링 → TF-IDF → Naive Bayes

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Baseline에서 사용한 것과 동일한 Train / Test 인덱스 사용
# X_train, X_test, y_train, y_test는 실습 2에서 만든 값 그대로 사용

# 2. 원본 df에서 같은 인덱스의 정제 텍스트 가져오기
train_texts = df.loc[X_train.index, "상품명_정제"]
test_texts = df.loc[X_test.index, "상품명_정제"]

# 3. TF-IDF 벡터화
# 훈련 데이터에만 fit
improved_vectorizer = TfidfVectorizer()

X_train_improved = improved_vectorizer.fit_transform(train_texts)
X_test_improved = improved_vectorizer.transform(test_texts)

# 4. Naive Bayes 학습
improved_model = MultinomialNB()
improved_model.fit(X_train_improved, y_train)

# 5. 예측
improved_pred = improved_model.predict(X_test_improved)

# 6. 평가
improved_accuracy = accuracy_score(y_test, improved_pred)
improved_macro_f1 = f1_score(
    y_test,
    improved_pred,
    average="macro"
)

improved_correct = (y_test == improved_pred).sum()
total_count = len(y_test)

print(f"Improved Accuracy: {improved_accuracy:.4f}")
print(f"Improved Accuracy(%): {improved_accuracy * 100:.2f}%")
print(f"Macro F1: {improved_macro_f1:.4f}")
print(f"정답: {improved_correct} / {total_count}")

print("\nClassification Report")
print(
    classification_report(
        y_test,
        improved_pred,
        zero_division=0
    )
)

# 7. Baseline과 비교
print("\nBefore / After")

print(
    f"Baseline Accuracy : {baseline_accuracy * 100:.2f}%"
)
print(
    f"Improved Accuracy : {improved_accuracy * 100:.2f}%"
)

print(
    f"Accuracy 변화     : "
    f"{(improved_accuracy - baseline_accuracy) * 100:.2f}%p"
)

print(
    f"Baseline Macro F1 : {baseline_macro_f1:.4f}"
)
print(
    f"Improved Macro F1 : {improved_macro_f1:.4f}"
)

print(
    f"정답 변화         : "
    f"{correct_count}권 → {improved_correct}권"
)

Improved Accuracy: 0.5789
Improved Accuracy(%): 57.89%
Macro F1: 0.4910
정답: 657 / 1135

Classification Report
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135


Before / After
Baseline Accuracy : 53.74%
Improved Accuracy : 57.89%
Accuracy 변화     : 4.14%p
Baseline Macro F1 : 0.3885
Improved Macro F1 : 0.4910
정답 변화         : 610권 → 657권


### 실행 결과 요약

- Improved Accuracy: **57.89%**
- Baseline Accuracy: **53.74%**
- 정확도 변화: **+4.14%p**
- Improved Macro F1: **0.4910**
- Baseline Macro F1: **0.3885**
- Macro F1 변화: **+0.1025**
- 정답 수 변화: **610권 → 657권**
- 정답 증가: **47권**

- 분야별 주요 개선: `경제/경영` F1-score **0.72**, `인문` **0.58**, `컴퓨터/IT` **0.81**로 비교적 높은 성능을 보였다.
- 분야별 추가 개선: `가정/육아`, `건강`, `요리`, `취미/실용/스포츠`는 기존보다 recall이 개선되었지만 여전히 낮은 편이다.
- 주요 한계: `소설`, `시/에세이`처럼 제목만으로 분야를 구분하기 어려운 카테고리는 성능 개선 폭이 제한적이었다.
- 최종 결과: 형태소 분석과 단어 필터링을 적용한 뒤 Accuracy와 Macro F1이 모두 상승해 전처리 개선 효과가 확인되었다.

## 실습 6. 추천 결과의 문제 개선하기

 

이번에는 Chapter 05와 Chapter 06의 추천 기능을 다시 봅니다.

In [8]:
# 실습 6. 추천 로직 개선
# 같은 분야 + 최소 유사도 기준 + 결과 출력

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. 정제된 제목으로 TF-IDF 생성
recommend_vectorizer = TfidfVectorizer()

title_matrix_improved = recommend_vectorizer.fit_transform(
    df["상품명_정제"]
)

# 2. 추천 함수
def recommend_books_improved(
    df,
    title_matrix,
    selected_index,
    top_n=5,
    min_similarity=0.1
):
    selected_category = df.loc[selected_index, "분야"]

    # 같은 분야 도서만 후보로 선택
    candidate_indices = df.index[
        df["분야"] == selected_category
    ].tolist()

    # 자기 자신 제외
    candidate_indices = [
        idx for idx in candidate_indices
        if idx != selected_index
    ]

    if not candidate_indices:
        return pd.DataFrame()

    # 선택 도서 벡터
    selected_vector = title_matrix[selected_index]

    # 후보 도서 벡터
    candidate_matrix = title_matrix[candidate_indices]

    # 코사인 유사도 계산
    similarities = cosine_similarity(
        selected_vector,
        candidate_matrix
    ).ravel()

    # 결과 데이터프레임
    result = df.loc[
        candidate_indices,
        ["상품명", "인물", "출판사", "분야"]
    ].copy()

    result["similarity"] = similarities

    # 최소 유사도 이상만 남기기
    result = result[
        result["similarity"] >= min_similarity
    ]

    # 유사도 높은 순 정렬
    result = (
        result
        .sort_values("similarity", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return result


# 3. 확인할 도서 제목
selected_title = "소년이 온다"

matched = df.index[
    df["상품명"] == selected_title
].tolist()

# 4. 결과 출력
if len(matched) == 0:
    print(f"'{selected_title}' 도서를 찾지 못했습니다.")

else:
    selected_index = matched[0]

    recommendations = recommend_books_improved(
        df=df,
        title_matrix=title_matrix_improved,
        selected_index=selected_index,
        top_n=5,
        min_similarity=0.1
    )

    print("선택 도서:", df.loc[selected_index, "상품명"])
    print("분야:", df.loc[selected_index, "분야"])

    if recommendations.empty:
        print("\n현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.")
    else:
        print("\n추천 결과")
        display(recommendations)

선택 도서: 소년이 온다
분야: 소설

추천 결과


,상품명,인물,출판사,분야,similarity
0,소년이로,편혜영,문학과지성사,소설,1.000000
1,바다에서 온 소년,개럿 카,북파머스,소설,0.722145


### 실행 결과 요약

- 선택 도서: `소년이 온다`
- 선택 분야: `소설`
- 추천 기준: 같은 분야 도서 중 최소 유사도 `0.1` 이상인 도서만 추천
- 추천 결과 수: **2권**
- 추천 도서 1: `소년이로` / 유사도 **1.000000**
- 추천 도서 2: `바다에서 온 소년` / 유사도 **0.722145**
- 주요 개선 결과: 유사도 기준을 적용해 의미 없는 0점 추천은 제거했고, 실제 제목 토큰이 겹치는 도서만 결과에 남았다.
- 해석: 추천 수가 5권보다 적더라도 억지로 채우지 않고, 근거가 있는 도서만 보여주는 방식으로 개선되었다.
- 한계: `소년이로`처럼 제목 토큰이 거의 같으면 유사도가 1.0까지 나올 수 있으므로, 제목만 사용하는 TF-IDF 추천에는 여전히 한계가 있다.